# Exercise 5.3 : $\int_0^\infty dx~ e^{-x^2}$ by various change of variables

### What you will learn

This notebook is just an exercise. We:
- compute $\int_0^\infty dx e^{-x^2}, $ using two changes of variables ($x = \frac{z}{1 - z}$ and $x = \tan z$), combined with three numerical methods (the trapezoidal rule, the trapezoidal rule with Euler–Maclaurin correction, and Gaussian quadrature);  
- compare the resulting $2 \times 3 = 6$ outcomes to see how they differ for this particular integral.

## Task
Compute $I=\int_0^\infty dx~ e^{-x^2}$ using the following two changes of variables:  
 - $x=z/(1-z)$
 - $x=\tan z$

For the numerical integration, apply each of the following three methods:      
 - the trapezoidal rule
 - the trapezoidal rule with the Euler-Maclauring correction $\frac{1}{12}h^2(f'(b)-f'(a))$
 - Gaussian quadrature

In total, we consider 2x3=6 combinations: two ways of variable transformation and three integration methods.   

For reference, the true value of the integral is $\sqrt{\pi}/2$, which is:

In [1]:
from math import exp, sqrt, pi, cos, tan
print(sqrt(pi)/2)

0.8862269254527579


### Function for each integral method

In [2]:
import numpy as np
import os, sys
sys.path.append(os.path.abspath(".."))
from utils import gaussxw

def trapezoid(f, a, b, N):
    "returns integral value of f from a to b using trapezoid approximation"
    assert a<b
    h = (b-a)/N
    result = 0.
    for k in range(1,N+1):
        xk = a + h*k
        result += h*f(xk)
    result += 0.5*h*(f(a) - f(b))
    return result

def euler_maclaurin(f, df, a, b, N):
    assert a<b
    h = (b-a)/N  
    correction = 1/12*h*h*(df(a) - df(b))
    return trapezoid(f, a, b, N) + correction

def gauss(f, a, b, N):
    "returns integral value of f from a to b using Gaussian quadrature"
    assert a<b
    x, w = gaussxw.gaussxwab(N, a, b)
    result = 0.
    for k in range(N):
        result += w[k] * f(x[k])
    return float(result)

### $x=z/(1-z)$  
First we consider the case of using transformation $x=z/(1-z)$.  In this case,   
$I=\int_0^\infty dx~ e^{-x^2} = \int_0^1 dz~ g(z)$  with $g(z)=\frac{1}{(1-z)^2}\exp\left( - \frac{z^2}{(1-z)^2} \right)$.  
The derivative is $g'(z)=2\frac{z^3-3z+1}{(1-z)^5}\exp(-\frac{z^2}{(1-z)^2})$, which is used for euler-maclaurin correction.   

In [3]:
def g(z):
    return exp(- z**2/(1-z)**2) / (1-z)**2 if z<1. else 0.  # we treat the case of z=1 separately to avoid zero division 

def dg(z):
    return exp(-z**2/(1-z)**2)*2*(z**2-3*z+1)/(1-z)**5 if z<1. else 0.  # we treat the case of z=1 separately to avoid zero division 

print(f"true value                 : {sqrt(pi)/2}")
value_trapz = trapezoid(g, 0, 1, N=100)
print(f"integral value by trapezoid      : {value_trapz}, its relative error is {abs(value_trapz-sqrt(pi)/2)/(sqrt(pi)/2)}")
value_em = euler_maclaurin(g, dg, 0, 1, N=100)
print(f"integral value by euler-maclaurin: {value_em}, its relative error is {abs(value_em-sqrt(pi)/2)/(sqrt(pi)/2)}")
value_gauss = gauss(g, 0, 1, N=100)
print(f"integral value by gauss          : {value_gauss}, its relative error is {abs(value_gauss-sqrt(pi)/2)/(sqrt(pi)/2)}")

true value                 : 0.8862269254527579
integral value by trapezoid      : 0.8862102587861348, its relative error is 1.8806319402481906e-05
integral value by euler-maclaurin: 0.8862269254528015, its relative error is 4.9107899247218366e-14
integral value by gauss          : 0.886226925452758, its relative error is 1.2527525318167949e-16


Thus, we see that Gauss quadrature is the most accurate, the trapezoidal rule with the Euler-Maclauin correction is the second most accurate, and the trapezoidal rule alone is the least accurate, as generally expected.  The result of Gauss quadrature reaches machine precision (float64).  

### $x=\tan(z)$    
Next we consider the case of using transformation $x=\tan(z)$.  In this case, since $dx = dz/\cos^2(z)$,   
$I=\int_0^\infty dx~ e^{-x^2} = \int_0^{\pi/2} dz~ g(z)$  with $g(z)=\frac{1}{\cos^2(z)}\exp(-\tan^2(z))$.  
The derivative is $g'(z)=-2\frac{\tan^3(z)}{\cos^2(z)}\exp(-\tan^2(z))$, which we use for euler-maclaurin correction.   

In [4]:
def g(z):
    #assert 0<=z<=pi/2
    eps = np.finfo(np.float64).eps
    return exp(-tan(z)**2) / cos(z)**2 if z<pi/2-eps else 0.  # we treat the case of z=pi/2 separately to avoid zero division 

def dg(z):
    assert 0<=z<=pi/2
    eps = np.finfo(np.float64).eps
    return -2*exp(-tan(z)**2) * tan(z)**3 / cos(z)**2 if z<pi/2-eps else 0.  # we treat the case of z=pi/2 separately to avoid zero division 


value_trapz = trapezoid(g, 0, pi/2, N=100)
print(f"integral value by trapezoid      : {value_trapz}, its relative error is {abs(value_trapz-sqrt(pi)/2)/(sqrt(pi)/2)}")
value_em = euler_maclaurin(g, dg, 0, pi/2, N=100)
print(f"integral value by euler-maclaurin: {value_em}, its relative error is {abs(value_em-sqrt(pi)/2)/(sqrt(pi)/2)}")
value_gauss = gauss(g, 0, pi/2, N=100)
print(f"integral value by gauss          : {value_gauss}, its relative error is {abs(value_gauss-sqrt(pi)/2)/(sqrt(pi)/2)}")

integral value by trapezoid      : 0.8862269254527577, its relative error is 2.5055050636335897e-16
integral value by euler-maclaurin: 0.8862269254527577, its relative error is 2.5055050636335897e-16
integral value by gauss          : 0.8862269254527583, its relative error is 3.758257595450385e-16


We see that, in the case of $x=\tan(z)$, even the trapezoidal rule with $N=100$ achieves machine precision.  (I do not know why $x=\tan(z)$ gives so precise result for this particular integral... It may be a coincidence, or there may be an underlying reason.) 